# `common.ipynb` — konstanta & fungsi bersama

Notebook ini **tidak dijalankan sendiri**. Setiap notebook pipeline memuatnya di sel pertamanya:

```python
%run ./common.ipynb
```

Semua yang didefinisikan di sini — path, konstanta pertandingan, utilitas, serta penulisan, validasi dan
penilaian submission — lalu tersedia di notebook pemanggil. Dengan begitu tidak ada file `.py` terpisah,
dan satu perubahan di sini berlaku untuk semua tahap.

| Bagian | Isi |
|---|---|
| 1 | Import |
| 2 | Path proyek & repo resmi |
| 3 | Konstanta pertandingan |
| 4 | Lokasi aset data |
| 5 | Utilitas umum |
| 6 | Submission: tulis, validasi, skor publik |
| 7 | Pembaca metadata ATD & Smart Tagging |
| 8 | Memuat artefak antara (`cache/`) |

**Aturan untuk isi notebook ini**
- Deterministik: tidak ada keacakan tanpa seed, tidak bergantung pada urutan dict/set.
- Tidak mencetak data per frame — hanya agregat (lihat aturan privasi di `docs/RENCANA.md`).
- Fungsi yang dipakai lebih dari satu notebook pindah ke sini; yang khusus satu tahap tetap di notebook tahap itu.

## 1. Import

Pipeline sengaja hanya memakai pustaka standar + `numpy` + `pandas` (sama seperti scorer resmi), supaya
mudah direproduksi. Pustaka lain hanya ditambahkan bila ablation membuktikan manfaatnya.

In [ ]:
import hashlib
import json
import os
import re
import sys
import time
import xml.etree.ElementTree as ET
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

np.set_printoptions(suppress=True, linewidth=120)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 2. Path proyek & repo resmi

| Nama | Arti |
|---|---|
| `ROOT` | akar repo — dicari ke atas dari folder kerja sampai ketemu `notebooks/common.ipynb`, jadi notebook bisa dijalankan dari Jupyter maupun `nbconvert` |
| `DATA_DIR` | folder berisi `tactical/` dan `broadcast/`. Default `ROOT`; bisa dipindah lewat variabel lingkungan `MTAH_DATA_DIR` (konvensi resmi `games.py`) |
| `CACHE` | artefak antara (gitignored) |
| `REPORTS` | laporan agregat tiap notebook, `cache/reports/<notebook>.txt/.json` |
| `SUBMISSIONS` | file CSV submission (gitignored) |
| `OFFICIAL` | repo resmi sebagai git submodule — sumber `games.py`, scorer, dan GT publik |

Kode resmi **dipakai langsung, tidak disalin**, supaya validasi dan skor lokal identik dengan milik panitia.

In [ ]:
def _find_root(start):
    for folder in [start, *start.parents]:
        if (folder / "notebooks" / "common.ipynb").is_file():
            return folder
    raise FileNotFoundError("Akar repo tidak ditemukan dari {}".format(start))


ROOT = _find_root(Path.cwd().resolve())
DATA_DIR = Path(os.environ.get("MTAH_DATA_DIR") or ROOT)
CACHE = ROOT / "cache"
REPORTS = CACHE / "reports"
SUBMISSIONS = ROOT / "submissions"
OFFICIAL = ROOT / "external" / "more-than-a-hack-2027"
GT_PUBLIC = OFFICIAL / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"

if not (OFFICIAL / "games.py").is_file():
    raise FileNotFoundError(
        "Repo resmi belum ada di {}. Jalankan: git submodule update --init".format(OFFICIAL))

for _folder in (CACHE, REPORTS, SUBMISSIONS):
    _folder.mkdir(parents=True, exist_ok=True)

# games.py membaca MTAH_DATA_DIR saat di-import, jadi harus di-set sebelum import.
os.environ["MTAH_DATA_DIR"] = str(DATA_DIR)
for _p in (OFFICIAL / "evaluation", OFFICIAL):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import games                      # noqa: E402  konstanta & nama aset per pertandingan (resmi)
import possession_metrics as pm   # noqa: E402  definisi semua metrik (resmi)

## 3. Konstanta pertandingan

Semua angka diambil dari `games.py` atau yang **dipublikasikan** di DATA.md/FAQ.md — tidak ada yang berasal
dari mengintip ground truth.

| Konstanta | Nilai | Sumber |
|---|---|---|
| `FPS` | 25 | `games.py` |
| `N_FRAMES` | 199.319 — jumlah baris submission | `games.py` (video tactical) |
| `PITCH_L × PITCH_W` | 105,3 × 68,0 m — untuk mengubah koordinat 0–1 ke meter | `games.py` |
| `HALVES` | babak 1: 11.185–90.651, babak 2: 110.808–190.197 (frame video, inklusif) | `games.half_video_blocks` |
| `KICKOFF[2]` | 113.742 — babak 2 dinilai sejak 110.808 tapi baru kick-off di sini, jadi 2.934 frame pertamanya pasti `D` | FAQ.md |
| `HALF`, `GRADED` | array per frame: babak (0 = tidak dinilai) dan topeng frame yang dinilai | turunan |
| `TEAM_NAME` | nama tim → nama file CSV & PDF (SUBMISSION.md) | **isi di Fase 0.1** |

In [ ]:
TEAM_NAME = "NAMA_TIM"   # TODO Fase 0.1: ganti dengan nama tim resmi

GAME = games.GAME                                   # "BAR-ATH"
FPS = games.GAMES[GAME]["fps"]                      # 25
N_FRAMES = games.source_frames("tactical", GAME)    # 199319
N_FRAMES_BROADCAST = games.source_frames("broadcast", GAME)
PITCH_L, PITCH_W = games.GAMES[GAME]["pitch"]       # 105.3, 68.0

HALVES = {h: (a, b) for h, a, b in games.half_video_blocks(GAME)}
KICKOFF = {1: HALVES[1][0], 2: 113742}

HALF = np.zeros(N_FRAMES, dtype=np.int8)
for _h, (_a, _b) in HALVES.items():
    HALF[_a:_b + 1] = _h
GRADED = HALF > 0
N_GRADED = int(GRADED.sum())

assert N_GRADED == 158857, N_GRADED

## 4. Lokasi aset data

`games.asset_paths()` membangun nama file standar. Nama file unduhan tidak selalu persis sama (contoh: video
kita bernama `BAR-ATH_tactical-002.mp4` dan `BAR-ATH_broadcast-001.mp4`), jadi `asset_path()` mencoba nama standar dulu, lalu
pola `<stem>*<akhiran>` bila hanya ada **satu** kecocokan. File data tidak diganti namanya.

- `asset_path(name, source)` → `Path` atau `None` bila aset tidak ada.
- `require_asset(name, source)` → `Path`, atau error yang jelas bila tidak ada.

`name` ∈ `video`, `atd_meta`, `atd_raw`, `homography`, `smart_tagging` (yang terakhir hanya untuk tactical).

In [ ]:
def asset_path(name, source="tactical"):
    """Path aset `name` untuk `source`, atau None bila tidak ada."""
    standard = Path(games.asset_paths(GAME, str(DATA_DIR), source)[name])
    if standard.is_file():
        return standard
    suffix = games.SUFFIXES[name]
    stem = standard.name[: -len(suffix)]
    matches = sorted(standard.parent.glob(stem + "*" + suffix))
    return matches[0] if len(matches) == 1 else None


def require_asset(name, source="tactical"):
    path = asset_path(name, source)
    if path is None:
        raise FileNotFoundError("Aset {}/{} tidak ditemukan di {}".format(source, name, DATA_DIR / source))
    return path

## 5. Utilitas umum

| Fungsi | Guna |
|---|---|
| `runs(mask)` | run `True` berurutan → `(start, end_eksklusif)`; dasar semua logika segmen |
| `Report(nama)` | laporan agregat: `rep("teks")` mencetak sekaligus mencatat, `rep.table(df)` untuk tabel, `rep.save(extra)` menyimpan ke `cache/reports/` — supaya hasil antar-run bisa dibandingkan tanpa membuka notebook |
| `sha256(path)` | sidik jari file (data masukan dan CSV submission) |
| `timer(label)` | `with timer("..."):` mencetak lama eksekusi |
| `segments_of(labels)` | label per frame → tabel segmen konstan (`start`, `end` eksklusif, `label`, `dur_s`) |
| `auc(pos, neg)` | daya beda satu skor: peluang skor contoh positif > negatif (0,5 = tak informatif, 1 = sempurna). Tanpa scikit-learn, lewat peringkat |

In [ ]:
def runs(mask):
    """Run True berurutan pada array 1-D -> (start, end_eksklusif) sebagai dua array int."""
    d = np.diff(np.concatenate(([0], np.asarray(mask).astype(np.int8), [0])))
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1)


class Report:
    """Laporan agregat satu notebook: dicetak sekaligus disimpan ke cache/reports/<nama>.txt."""

    def __init__(self, name):
        self.name, self.lines = name, []

    def __call__(self, text=""):
        print(text)
        self.lines.append(str(text))

    def table(self, df, title=None):
        if title:
            self(title)
        self(df.to_string())

    def save(self, extra=None):
        txt = REPORTS / (self.name + ".txt")
        txt.write_text("\n".join(self.lines) + "\n", encoding="utf-8")
        if extra is not None:
            (REPORTS / (self.name + ".json")).write_text(
                json.dumps(extra, indent=2, default=str), encoding="utf-8")
        print("\nlaporan disimpan: {}".format(txt.relative_to(ROOT)))


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


@contextmanager
def timer(label):
    t0 = time.time()
    yield
    print("[{}] {:.1f} s".format(label, time.time() - t0))


def segments_of(labels, offset=0):
    """Label per frame -> DataFrame segmen konstan. `offset` = frame video elemen pertama."""
    labels = np.asarray(labels)
    change = np.flatnonzero(labels[1:] != labels[:-1]) + 1
    starts = np.r_[0, change]
    ends = np.r_[change, len(labels)]
    return pd.DataFrame({"start": starts + offset, "end": ends + offset, "label": labels[starts],
                         "dur_s": (ends - starts) / FPS})


def auc(pos, neg):
    """P(skor positif > skor negatif), seri dihitung setengah. NaN diabaikan; NaN bila sampel < 5."""
    pos = np.asarray(pos, dtype=float)
    neg = np.asarray(neg, dtype=float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if len(pos) < 5 or len(neg) < 5:
        return float("nan")
    ranks = pd.Series(np.concatenate([pos, neg])).rank().to_numpy()
    return float((ranks[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))

## 6. Submission: tulis, validasi, skor publik

Kontrak file (SUBMISSION.md): header `frame,label`, tepat **199.319** baris, `frame` = 0…199318 berurutan,
label ∈ `H` `A` `D` `X`, tanpa sel kosong.

| Fungsi | Guna |
|---|---|
| `write_submission(labels, name)` | tulis CSV dengan akhir baris `\n` yang tetap → `(path, sha256)`. Hash dipakai untuk membuktikan reproduksi **persis** |
| `validate_file(path)` | **memakai validasi resmi** (`pm.validate_submission`), jadi lulus di sini = lulus di scorer. Juga meringkas jumlah label dan possession |
| `score_public(path)` | metrik utama terhadap GT publik 5 menit |

> ⚠️ `score_public` hanya untuk **cek kebenaran pipeline** (frame selaras, label bermakna benar). Jangan
> dipakai untuk memilih versi atau parameter — 17 possession terlalu sedikit (EVALUATION.md, dan §5 di
> `docs/RENCANA.md`).

In [ ]:
LABELS = np.array(["H", "A", "D", "X"])


def write_submission(labels, name, folder=None):
    labels = np.asarray(labels).astype(str)
    if labels.shape != (N_FRAMES,):
        raise ValueError("butuh {} label, dapat {}".format(N_FRAMES, labels.shape))
    unknown = sorted(set(np.unique(labels).tolist()) - set(LABELS.tolist()))
    if unknown:
        raise ValueError("label tidak dikenal: {}".format(unknown))
    path = Path(folder or SUBMISSIONS) / "{}.csv".format(name)
    with open(path, "w", newline="\n", encoding="utf-8") as handle:
        handle.write("frame,label\n")
        handle.writelines("{},{}\n".format(i, lab) for i, lab in enumerate(labels))
    return path, sha256(path)


def validate_file(path):
    """(lulus?, ringkasan) memakai validasi resmi."""
    table = pm.read_label_csv(str(path))
    reason = pm.validate_submission(table, N_FRAMES)
    if reason:
        return False, {"reason": reason}
    codes = pm.label_codes(table)
    spells = pm.possession_spells(codes, pm.valid_runs(codes))
    alive = int(np.sum((codes == pm.H) | (codes == pm.A)))
    summary = {lab: int(np.sum(codes == i)) for i, lab in enumerate(pm.LABELS)}
    summary["n_possessions"] = len(spells)
    summary["mean_possession_s"] = round(alive / FPS / max(len(spells), 1), 2)
    return True, summary


_GT_PUBLIC_CODES = None


def score_public(path, bootstrap_n=0):
    """Metrik utama terhadap GT publik (5 menit). HANYA untuk cek kebenaran pipeline."""
    global _GT_PUBLIC_CODES
    if _GT_PUBLIC_CODES is None:
        _GT_PUBLIC_CODES = pm.label_codes(pm.read_label_csv(str(GT_PUBLIC)))
    pred = pm.label_codes(pm.read_label_csv(str(path)))
    result = pm.evaluate(_GT_PUBLIC_CODES, pred, bootstrap_n=bootstrap_n)
    keys = ["spell_f1_mean", "macro_f1", "n_pred_spells", "n_gt_spells", "mean_spell_duration_s",
            "alive_dead_f1", "team_acc_gt_alive", "spell_bootstrap_low", "spell_bootstrap_high"]
    return {k: result.get(k) for k in keys}

## 7. Pembaca metadata ATD & Smart Tagging

Dipakai oleh `00_setup_cek_data` (untuk dicek) dan `01_preprocess` (untuk dipakai), jadi tinggal di sini.

**`parse_atd_meta(source)`** membaca XML FIFA/EPTS. File ini memetakan setiap **slot track** (bukan pemain!)
ke sebuah tim. Kembaliannya:

| Kunci | Isi |
|---|---|
| `home_id`, `away_id`, `home_name`, `away_name` | tim; kandang = tim **pertama** di XML (konvensi baseline resmi & kloppy) |
| `local_team_id` | `idLocalTeam` pada elemen `Score` — cara kedua menentukan kandang, untuk dicek silang |
| `slot_ids` | id slot dalam urutan elemen `Player` = urutan grup di setiap baris ATD |
| `channel_order` | urutan kanal `player<id>_x` di definisi data — harus sama dengan `slot_ids` |
| `slot_is_home` | array bool per slot |
| `attack_h1` | arah serang tim kandang di babak 1 (`left_to_right` / `right_to_left`) |

Kanal bola di XML adalah `x`, `y`, `s` → kolom ketiga bola adalah **kecepatan**, bukan ketinggian.

**`load_smart_tagging(meta)`** membaca SportsCode XML jadi DataFrame satu baris per instans: `id`, `code`
(huruf besar), `start_s`/`end_s`, `start_f`/`end_f` = `round(detik × 25)` (DATA.md), satu kolom per grup label
(`Team`, `Half`, `Type`, …), dan `side` ∈ `H`/`A`/`NA` (`?` = nama tim tak dikenal → harus diselidiki).

In [ ]:
def _local(tag):
    return tag.split("}")[-1]


def _child_text(el, *names):
    for child in el:
        if _local(child.tag).lower() in names:
            return (child.text or "").strip()
    return ""


def parse_atd_meta(source="tactical"):
    path = require_asset("atd_meta", source)
    root = ET.parse(path).getroot()
    teams = [el for el in root.iter() if _local(el.tag) == "Team" and el.get("id")]
    players = [el for el in root.iter() if _local(el.tag) == "Player" and el.get("id") and el.get("teamId")]
    local_ids = [el.get("idLocalTeam") for el in root.iter() if _local(el.tag) == "Score" and el.get("idLocalTeam")]
    home_id = teams[0].get("id") if teams else None
    away_id = teams[1].get("id") if len(teams) > 1 else None

    attack_h1 = None
    for el in (teams[0].iter() if teams else []):
        if _local(el.tag) == "ProviderParameter" and _child_text(el, "name", "n") == "attack_direction_first_half":
            attack_h1 = _child_text(el, "value")

    return {
        "n_teams": len(teams),
        "home_id": home_id,
        "away_id": away_id,
        "home_name": _child_text(teams[0], "name", "n") if teams else None,
        "away_name": _child_text(teams[1], "name", "n") if len(teams) > 1 else None,
        "local_team_id": local_ids[0] if local_ids else None,
        "slot_ids": [int(el.get("id")) for el in players],
        "channel_order": [int(i) for i in re.findall(r'playerChannelId="player(\d+)_x"',
                                                     path.read_text(encoding="utf-8"))],
        "slot_is_home": np.array([el.get("teamId") == home_id for el in players]),
        "attack_h1": attack_h1,
    }


def load_smart_tagging(meta):
    rows = []
    for ins in ET.parse(require_asset("smart_tagging")).getroot().iter("instance"):
        row = {"id": int(ins.findtext("ID")), "code": (ins.findtext("code") or "").strip().upper(),
               "start_s": float(ins.findtext("start")), "end_s": float(ins.findtext("end"))}
        for lab in ins.findall("label"):
            row[(lab.findtext("group") or "").strip()] = (lab.findtext("text") or "").strip()
        rows.append(row)
    tags = pd.DataFrame(rows)
    tags["start_f"] = (tags.start_s * FPS).round().astype(int)
    tags["end_f"] = (tags.end_s * FPS).round().astype(int)

    names = {(meta["home_name"] or "").lower(): "H", (meta["away_name"] or "").lower(): "A"}
    team = tags["Team"].fillna("N/A") if "Team" in tags else pd.Series("N/A", index=tags.index)
    tags["side"] = team.str.lower().map(lambda v: names.get(v, "NA" if v in ("n/a", "") else "?"))
    return tags.sort_values(["start_f", "id"]).reset_index(drop=True)

## 8. Memuat artefak antara

Notebook tahap berikutnya tidak membaca data mentah lagi — mereka memuat hasil `01_preprocess`:

| Fungsi | Isi |
|---|---|
| `load_prep()` | `cache/prep.npz` sebagai `dict` array per frame video (kunci dijelaskan di `01_preprocess`) |
| `load_tags()` | `cache/tags.csv` — Smart Tagging dalam frame video; sel kosong tetap string kosong, `N/A` tetap `N/A` |

Keduanya gagal dengan pesan jelas bila `01_preprocess` belum dijalankan.

In [ ]:
def load_prep():
    path = CACHE / "prep.npz"
    if not path.is_file():
        raise FileNotFoundError("{} belum ada — jalankan 01_preprocess.ipynb dulu".format(path))
    with np.load(path) as z:
        return {k: z[k] for k in z.files}


def load_tags():
    path = CACHE / "tags.csv"
    if not path.is_file():
        raise FileNotFoundError("{} belum ada — jalankan 01_preprocess.ipynb dulu".format(path))
    return pd.read_csv(path, keep_default_na=False)

In [ ]:
print("common.ipynb dimuat | ROOT = {} | data = {} | {} frame, {} dinilai".format(
    ROOT, DATA_DIR, N_FRAMES, N_GRADED))